# 🔍 실시간 DB 업데이트 기반  시스템

### 🧩 단계 1: 데이터 준비
- 엑셀 파일에 있는 제품 설명과 정보를 불러옵니다.
- 각 행(row)을 하나의 문서처럼 구성하고, 요약 → 상세 순으로 정리합니다.

### 🧠 단계 2: 문서를 '벡터'로 저장
- 문서들을 수치 벡터로 바꿔서 Chroma라는 데이터베이스에 저장합니다.
- 이 벡터는 AI가 의미를 이해할 수 있게 만드는 일종의 언어 번역입니다.

### 🔄 단계 3: 실시간 DB 업데이트
- 새로운 엑셀 파일을 넣으면, 기존 DB에 문서를 자동으로 추가합니다.
- 즉, 매번 새로 만들지 않고 기존 지식에 ‘덧붙이기’만 하면 됩니다.

### 💬 단계 4: 질문 → 문서 검색 → 답변
- 사용자가 질문을 하면, AI는 관련 문서 5개를 찾아보고 내용을 요약해 답변합니다.
- 답변에는 관련된 제품의 행 번호도 함께 표시됩니다.

✅ 이렇게 하면, 항상 최신 정보를 반영하는 스마트한 AI 비서가 완성됩니다!


In [ ]:
!pip install -U langchain-community -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.8 MB/s eta 0:00:00


In [ ]:
!pip install pandas openpyxl langchain langchain-openai chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 1.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.4/70.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 119.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.2/284.2 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.5/118.5 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.2/196.2 kB 17.3 MB/s e

In [ ]:
# 필요한 모듈 불러오기
import openai  # [포인트] OpenAI의 GPT 모델을 사용하기 위한 공식 라이브러리입니다.
import os      # [포인트] 환경 변수 설정을 위해 사용하는 파이썬 내장 모듈입니다. API 키를 숨길 수 있어 보안상 유리합니다.
# API 키를 환경 변수에 저장하여 보안 강화
# [예시] 자물쇠로 잠긴 금고에 API 키를 보관하는 느낌입니다. 키를 코드에 직접 쓰지 않아 보안이 향상됩니다.
os.environ['OPENAI_API_KEY'] = ""  # [포인트] 실제 API 키를 환경 변수 OPENAI_API_KEY에 저장합니다.
openai.api_key = os.getenv('OPENAI_API_KEY')  # [포인트] 환경 변수에서 API 키를 불러와 openai 모듈에 등록합니다.


In [ ]:

# ✅ 모듈 불러오기
import openai  # OpenAI API 사용을 위한 모듈
import os  # 시스템 환경 설정용
import pandas as pd  # 엑셀 등 표 형식 데이터 처리
from langchain.docstore.document import Document  # 텍스트 문서를 담는 LangChain 문서 객체
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # OpenAI의 임베딩 및 챗 모델 로드
from langchain.vectorstores import Chroma  # 벡터 DB로 Chroma 사용
from langchain.prompts.chat import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate  # 대화형 프롬프트 구성 도구
from langchain.schema.runnable import RunnablePassthrough  # 입력 그대로 전달하는 중간 노드
from langchain.schema.output_parser import StrOutputParser  # 모델 응답을 문자열로 파싱

# ✅ 엑셀 파일 로드
df = pd.read_excel('/content/Sample.xlsx')  # 엑셀 파일을 판다스로 불러옴

# ✅ 문서 생성 (요약을 가장 먼저 배치)
docs = []
for i, row in df.iterrows():  # [포인트] 각 행(row)을 하나의 문서로 처리
    summary = f"요약: {row['설명 및 소개'] if pd.notnull(row['설명 및 소개']) else ''}"  # [포인트] 설명 컬럼을 요약 형태로 가장 앞에 배치
    detail = "\n".join([
        f"{col}: {row[col]}" for col in df.columns
        if col != '설명 및 소개' and pd.notnull(row[col])
    ])  # [포인트] 나머지 컬럼을 줄 단위로 정리
    full_text = f"{summary}\n{detail}"  # 요약 + 상세 내용을 하나의 문자열로 합침
    docs.append(Document(page_content=full_text, metadata={"row": i}))  # [포인트] row 번호를 메타데이터로 저장

# ✅ 확인용 출력
for i, doc in enumerate(docs[:3]):  # 앞의 3개 문서만 출력하여 확인
    print(f"📄 문서 {i+1}")
    print(doc.page_content)
    print("-" * 80)

# ✅ 벡터 DB 구축
embeddings = OpenAIEmbeddings()  # [포인트] 문서를 숫자 벡터로 변환하는 OpenAI 임베딩 사용
db = Chroma.from_documents(docs, embedding=embeddings, persist_directory="./chroma_db")  # [포인트] 변환된 문서를 Chroma DB에 저장
retriever = db.as_retriever(search_kwargs={"k": 5})  # [포인트] 관련 문서 5개를 검색할 수 있도록 설정


📄 문서 1
요약: 고성능 무선 이어폰입니다
제품명: 이어팟 프로
카테고리: 전자기기
가격: 199000
제조사: 사운드웨이브
--------------------------------------------------------------------------------
📄 문서 2
요약: 초경량 노트북입니다
제품명: 라이트북 X
카테고리: 컴퓨터
가격: 1290000
제조사: 컴퓨존
--------------------------------------------------------------------------------
📄 문서 3
요약: 천연 성분 샴푸입니다
제품명: 내추럴샴푸
카테고리: 생활용품
가격: 12000
제조사: 그린코스메틱
--------------------------------------------------------------------------------


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [ ]:
# ✅ 벡터 DB 업데이트 함수 (동일한 요약 강조 포함)
def update_vectordb_from_excel(file_path, db):  # [포인트] 새 엑셀 데이터를 벡터 DB에 추가하는 함수
    new_df = pd.read_excel(file_path)  # 새 엑셀 파일 로드
    new_docs = []
    for i, row in new_df.iterrows():  # 새 행마다 문서 구성
        summary = f"요약: {row['설명 및 소개'] if pd.notnull(row['설명 및 소개']) else ''}"  # 요약 먼저
        detail = "\n".join([
            f"{col}: {row[col]}" for col in new_df.columns
            if col != '설명 및 소개' and pd.notnull(row[col])
        ])
        full_text = f"{summary}\n{detail}"
        new_docs.append(Document(page_content=full_text, metadata={"row": i}))
    db.add_documents(new_docs)  # 벡터 DB에 새 문서 추가
    db.persist()  # DB 저장
    print(f"✅ {len(new_docs)}개의 문서가 벡터 DB에 추가되었습니다.")  # 결과 출력


In [ ]:

# ✅ 프롬프트 설정
system_prompt = """당신은 사용자 맞춤형 상품 정보 서비스플랫폼의 AI 비서입니다.
다음은 사용자의 질문에 답하기 위해 참고할 수 있는 데이터 문서입니다.

당신의 목표는:
- 사용자의 자연어 질의에 대해,
- 아래 문서들을 바탕으로 가장 관련도 높은 정보를 찾고,
- 명확하고 실용적인 한국어로 답변을 제공하며,
- 필요 시 제품명, 특징, 가격등을 제시하고,
- 해당 정보가 어느 행(row)에 있는지도 함께 출력하는 것입니다.

질문에 대해 아래의 순서로 답변하세요:
1. 문서 내용이 질문과 얼마나 관련이 있는지 짧게 평가하세요.
2. 질문에 대한 정확한 답변을 제공하세요.
3. 출처 정보를 `row id`로 표시하세요. (예: row: 17)

---
{summaries}
---
Answer:"""  # [포인트] AI 비서의 역할과 응답 방식 정의, row 번호까지 출처로 출력

messages = [
    SystemMessagePromptTemplate.from_template(system_prompt),  # 시스템 메시지 삽입 (AI 역할 안내)
    HumanMessagePromptTemplate.from_template("질문: {question}")  # 사용자의 질문 포맷 정의
]
prompt = ChatPromptTemplate.from_messages(messages)  # 전체 프롬프트 구조 결합

# ✅ LLM 체인 구성
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.1, max_tokens=4096)  #
chain = (
    {"summaries": retriever, "question": RunnablePassthrough()}  # [포인트] 질문과 관련 문서를 함께 넘김
    | prompt  # 프롬프트 형식에 맞게 입력 구성
    | llm  # GPT에 전달
    | StrOutputParser()  # [포인트] GPT 응답을 텍스트로 파싱
)


In [ ]:
# ✅ 테스트 쿼리 실행
query = "펫피더 스마트에 대해 설명해줘"
result = chain.invoke(query)
print(result)

1. 문서 내용이 질문과 관련이 없습니다. 제공된 문서에는 "펫피더 스마트"에 대한 정보가 포함되어 있지 않습니다.
2. 질문에 대한 정확한 답변을 제공할 수 없습니다. "펫피더 스마트"에 대한 정보는 제공된 데이터 문서에 없습니다.
3. 출처 정보를 제공할 수 없습니다.


In [ ]:
update_vectordb_from_excel("/content/Sample_updated.xlsx", db)


✅ 3개의 문서가 벡터 DB에 추가되었습니다.


/tmp/ipython-input-5-2709681007.py:14: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()  # DB 저장


In [ ]:
# ✅ 추가된 DB에 대한 쿼리 실행
query = "펫피더 스마트에 대해 설명해줘!"
result = chain.invoke(query)
print(result)

1. 문서 내용이 질문과 관련이 높습니다. 사용자가 언급한 "펫피더 스마트"에 대한 정보가 문서에 포함되어 있습니다.

2. 펫피더 스마트는 자동 급식 기능이 있는 반려동물용 식기입니다. 이 제품은 반려용품 카테고리에 속하며, 가격은 85,000원입니다. 제조사는 펫케어코리아입니다.

3. 출처 정보: row: 0


In [ ]:
# ✅ 테스트 쿼리 실행
query = "클린플랜트에 대해 설명해줘"
result = chain.invoke(query)
print(result)



1. 문서 내용이 질문과 관련이 높습니다. 클린플랜트에 대한 정보가 포함되어 있습니다.
2. 클린플랜트는 실내용 공기 정화 기능을 갖춘 스마트 화분입니다. 생활가전 카테고리에 속하며, 가격은 99,000원입니다. 제조사는 그로우가든입니다.
3. 출처 정보: row: 1
